In [1]:
!pip install -q langchain-community openai faiss-cpu chromadb tiktoken
!pip install -q google-generativeai pymupdf
!pip install -q langchain-chroma langchain-openai langgraph langchain-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 8.2 MB/s eta 0:00:

In [2]:
# ============================================================================
# PROCESADOR MÚLTIPLE DE PAPERS - VECTORES DUALES
# ============================================================================

import os
import re
import json
import pickle
import warnings
from typing import TypedDict, List, Dict, Any
from pathlib import Path

# PDF Processing
import fitz  # pymupdf

# LangChain Core
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain.schema import Document as LCDocument

# APIs externas
import google.generativeai as genai
from google.colab import userdata, drive

warnings.filterwarnings('ignore')

In [3]:
# ============================================================================
# CONFIGURACIÓN DE APIS Y RATE LIMITING PARA GEMINI
# ============================================================================

import time
from datetime import datetime, timedelta

# Montar Drive
print("📁 Montando Google Drive...")
drive.mount('/content/drive')

# Configurar APIs
print("🔑 Configurando APIs...")
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
genai.configure(api_key=userdata.get('GEMINI_API_KEY_1'))

# Inicializar modelos
print("🤖 Inicializando modelos...")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()
gemini_model = genai.GenerativeModel('gemini-2.0-flash')

# ============================================================================
# SISTEMA DE RATE LIMITING PARA GEMINI GRATUITO
# ============================================================================

class GeminiRateLimiter:
    """Sistema de rate limiting para API gratuita de Gemini"""

    def __init__(self):
        # Límites de API gratuita (con margen de seguridad)
        self.rpm_limit = 12  # 15 real, usamos 12 para seguridad
        self.tpm_limit = 800_000  # 1M real, usamos 800K para seguridad
        self.rpd_limit = 180  # 200 real, usamos 180 para seguridad

        # Contadores
        self.requests_this_minute = 0
        self.tokens_this_minute = 0
        self.requests_today = 0

        # Timestamps
        self.last_request_time = 0
        self.minute_start = time.time()
        self.day_start = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)

        print(f"🚦 Rate Limiter inicializado:")
        print(f"   📊 RPM: {self.rpm_limit}/min")
        print(f"   📊 TPM: {self.tpm_limit:,}/min")
        print(f"   📊 RPD: {self.rpd_limit}/día")

    def estimate_tokens(self, text: str) -> int:
        """Estima tokens para rate limiting (aproximación conservadora)"""
        return len(text) // 3  # Aproximación: 1 token ≈ 3 caracteres

    def wait_if_needed(self, estimated_tokens: int = 5000):
        """Espera si es necesario para respetar rate limits"""
        current_time = time.time()

        # Reset contadores por minuto si ha pasado un minuto
        if current_time - self.minute_start >= 60:
            self.requests_this_minute = 0
            self.tokens_this_minute = 0
            self.minute_start = current_time

        # Reset contador diario
        current_day = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
        if current_day > self.day_start:
            self.requests_today = 0
            self.day_start = current_day

        # Verificar límite diario
        if self.requests_today >= self.rpd_limit:
            wait_until_tomorrow = (self.day_start + timedelta(days=1) - datetime.now()).total_seconds()
            print(f"⏰ Límite diario alcanzado. Esperando hasta mañana ({wait_until_tomorrow/3600:.1f} horas)")
            time.sleep(wait_until_tomorrow)
            self.requests_today = 0
            self.day_start = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)

        # Verificar límites por minuto
        if (self.requests_this_minute >= self.rpm_limit or
            self.tokens_this_minute + estimated_tokens > self.tpm_limit):

            wait_time = 60 - (current_time - self.minute_start)
            if wait_time > 0:
                print(f"⏰ Rate limit: esperando {wait_time:.1f}s (RPM: {self.requests_this_minute}/{self.rpm_limit}, TPM: {self.tokens_this_minute:,}/{self.tpm_limit:,})")
                time.sleep(wait_time)
                self.requests_this_minute = 0
                self.tokens_this_minute = 0
                self.minute_start = time.time()

        # Delay mínimo entre requests (5 segundos para ser conservadores)
        min_interval = 5
        time_since_last = current_time - self.last_request_time
        if time_since_last < min_interval:
            wait_time = min_interval - time_since_last
            print(f"⏰ Delay: esperando {wait_time:.1f}s")
            time.sleep(wait_time)

        self.last_request_time = time.time()

    def record_request(self, tokens_used: int = 5000):
        """Registra una request realizada"""
        self.requests_this_minute += 1
        self.tokens_this_minute += tokens_used
        self.requests_today += 1

        print(f"📊 Request #{self.requests_today} (Min: {self.requests_this_minute}/{self.rpm_limit}, Tokens: {self.tokens_this_minute:,}/{self.tpm_limit:,})")

    def get_status(self) -> dict:
        """Retorna estado actual del rate limiter"""
        return {
            'requests_this_minute': self.requests_this_minute,
            'tokens_this_minute': self.tokens_this_minute,
            'requests_today': self.requests_today,
            'rpm_remaining': self.rpm_limit - self.requests_this_minute,
            'tpm_remaining': self.tpm_limit - self.tokens_this_minute,
            'rpd_remaining': self.rpd_limit - self.requests_today
        }

# Inicializar rate limiter global
rate_limiter = GeminiRateLimiter()

print("✅ APIs configuradas correctamente!")
print("✅ Modelos listos para usar:")
print(f"  - LLM: {llm.model_name}")
print(f"  - Embeddings: OpenAI")
print(f"  - Gemini: gemini-2.0-flash")

# Test rápido de conexiones
print("\n🧪 Probando conexiones...")
try:
    # Test OpenAI
    test_embedding = embeddings.embed_query("test")
    print("✅ OpenAI: Conectado")
except Exception as e:
    print(f"❌ OpenAI: Error - {e}")

try:
    # Test Gemini con rate limiting
    rate_limiter.wait_if_needed(100)
    test_response = gemini_model.generate_content("Di solo 'OK' si me recibes")
    rate_limiter.record_request(100)
    print(f"✅ Gemini: Conectado - Respuesta: {test_response.text.strip()}")
except Exception as e:
    print(f"❌ Gemini: Error - {e}")

print("\n🎯 Variables globales disponibles:")
print("  - llm")
print("  - embeddings")
print("  - gemini_model")
print("  - rate_limiter")
print("\n✅ Listo para ejecutar el procesamiento de papers con rate limiting!")

📁 Montando Google Drive...
Mounted at /content/drive
🔑 Configurando APIs...
🤖 Inicializando modelos...
🚦 Rate Limiter inicializado:
   📊 RPM: 12/min
   📊 TPM: 800,000/min
   📊 RPD: 180/día
✅ APIs configuradas correctamente!
✅ Modelos listos para usar:
  - LLM: gpt-4o-mini
  - Embeddings: OpenAI
  - Gemini: gemini-2.0-flash

🧪 Probando conexiones...
✅ OpenAI: Conectado
📊 Request #1 (Min: 1/12, Tokens: 100/800,000)
✅ Gemini: Conectado - Respuesta: OK

🎯 Variables globales disponibles:
  - llm
  - embeddings
  - gemini_model
  - rate_limiter

✅ Listo para ejecutar el procesamiento de papers con rate limiting!


In [4]:
# ============================================================================
# FUNCIONES DE PROCESAMIENTO CON RATE LIMITING
# ============================================================================

def sanitize_collection_name(name: str) -> str:
    """Sanitiza nombres para compatibilidad con Chroma"""
    # Remover caracteres especiales y reemplazar con guiones bajos
    sanitized = re.sub(r'[^a-zA-Z0-9._-]', '_', name)

    # Remover múltiples guiones bajos consecutivos
    sanitized = re.sub(r'_+', '_', sanitized)

    # Asegurar que comience y termine con alfanumérico
    sanitized = sanitized.strip('_.-')

    # Si está vacío o muy corto, usar nombre genérico
    if len(sanitized) < 3:
        sanitized = f"paper_{hash(name) % 10000}"

    # Limitar longitud a 50 caracteres para seguridad
    if len(sanitized) > 50:
        sanitized = sanitized[:50].rstrip('_.-')

    return sanitized

def extract_pdf_text(pdf_path: str) -> tuple:
    """Extrae texto del PDF con metadata básica"""
    try:
        doc = fitz.open(pdf_path)
        full_text = "\n".join([page.get_text() for page in doc])

        metadata = {
            'total_pages': len(doc),
            'total_chars': len(full_text),
            'filename': pdf_path,
            'paper_name': Path(pdf_path).stem
        }

        doc.close()
        return full_text, metadata
    except Exception as e:
        print(f"❌ Error procesando {pdf_path}: {e}")
        return None, None

def gemini_request_with_rate_limit(prompt: str, description: str = "") -> str:
    """Realiza request a Gemini con rate limiting"""
    estimated_tokens = rate_limiter.estimate_tokens(prompt)

    print(f"  🤖 {description} (≈{estimated_tokens:,} tokens)")

    # Rate limiting
    rate_limiter.wait_if_needed(estimated_tokens)

    # Realizar request
    response = gemini_model.generate_content(prompt)
    rate_limiter.record_request(estimated_tokens)

    return response.text

def extract_title_from_text(text: str, gemini_model) -> str:
    """Extrae el título del paper usando Gemini con rate limiting"""
    try:
        # Truncar texto para ahorrar tokens
        first_part = text[:2000]

        prompt = f"""Analiza el siguiente texto del inicio de un artículo científico y extrae únicamente el TÍTULO principal del paper.

Texto:
{first_part}

Instrucciones:
- Identifica y extrae solo el título principal del artículo
- NO incluyas nombres de autores, afiliaciones, resumen, o cualquier otro contenido
- Devuelve únicamente el título, sin comillas ni prefijos
- Si hay múltiples líneas que parecen título, combínalas en una sola línea
- Si no puedes identificar un título claro, devuelve "Título no identificado"

TÍTULO:"""

        response_text = gemini_request_with_rate_limit(prompt, "Extrayendo título")
        title = response_text.strip()

        # Limpiar el título de posibles prefijos
        title = re.sub(r'^(TÍTULO|Title|TITLE):\s*', '', title, flags=re.IGNORECASE)
        title = title.strip('"').strip("'").strip()

        if not title or len(title) < 5:
            return "Título no identificado"

        return title

    except Exception as e:
        print(f"⚠️ Error extrayendo título: {e}")
        return "Título no identificado"

def create_analysis_prompts_optimized(text: str) -> Dict[str, str]:
    """Crea prompts optimizados para rate limiting"""
    # Truncar texto si es muy largo para ahorrar tokens
    max_chars = 50000  # ~15k tokens aproximadamente
    if len(text) > max_chars:
        text = text[:max_chars] + "\n\n[TEXTO TRUNCADO PARA OPTIMIZACIÓN]..."

    base_text = f"Analiza el siguiente texto de un artículo científico:\n\n{text}"

    return {
        'sections': f"""{base_text}

Identifica y lista todos los encabezados de sección principales (como Introduction, Methods, Results, Discussion, Conclusion, etc.) en el orden que aparecen.

Formato de respuesta:
## SECCIONES IDENTIFICADAS:
- [Lista de secciones en orden]

Solo los nombres de las secciones, sin explicaciones adicionales.""",

        'references': f"""{base_text}

Identifica y extrae todas las referencias bibliográficas del artículo.

Formato de respuesta:
## REFERENCIAS BIBLIOGRÁFICAS:
- [Número] Referencia completa"""
    }

def parse_gemini_sections(response_text: str) -> List[str]:
    """Extrae secciones de respuesta de Gemini"""
    sections = []
    if "## SECCIONES IDENTIFICADAS:" in response_text:
        sections_text = response_text.split("## SECCIONES IDENTIFICADAS:")[1]
        if "## REFERENCIAS BIBLIOGRÁFICAS:" in sections_text:
            sections_text = sections_text.split("## REFERENCIAS BIBLIOGRÁFICAS:")[0]

        for line in sections_text.split('\n'):
            line = line.strip()
            if line.startswith('- '):
                section = line[2:].strip()
                if section:
                    sections.append(section)
    return sections

def parse_references_to_dict(text: str) -> dict:
    """Convierte referencias en diccionario {numero: texto}"""
    matches = list(re.finditer(r"- \[(\d+)\] (.+?)(?=(?:- \[\d+\])|\Z)", text, re.DOTALL))
    return {match.group(1).strip(): match.group(2).strip().replace("\n", " ").replace("  ", " ")
            for match in matches}

def extract_references(text: str) -> List[str]:
    """Extrae números de referencias citadas del texto"""
    raw_refs = re.findall(r"\[([^\[\]]+?)\]", text)
    final_refs = set()

    for group in raw_refs:
        parts = [p.strip() for p in group.split(',')]
        for part in parts:
            if '–' in part or '-' in part:
                sep = '–' if '–' in part else '-'
                try:
                    start, end = map(int, part.split(sep))
                    final_refs.update(str(i) for i in range(start, end + 1))
                except ValueError:
                    continue
            else:
                if part.isdigit():
                    final_refs.add(part)

    return sorted(final_refs, key=int)

def split_text_by_sections(full_text: str, section_titles: List[str], resolved_references: dict) -> List[Dict]:
    """Divide el texto por secciones y asocia referencias"""
    sections_data = []
    text_with_markers = full_text + "\n## END_OF_DOCUMENT##"

    # Crear patrón regex para títulos de sección
    escaped_titles = [re.escape(title) for title in section_titles]
    section_pattern = '|'.join([f'^{title}' for title in escaped_titles])
    end_pattern = r'^## END_OF_DOCUMENT##'
    combined_pattern = f'(?m){section_pattern}|{end_pattern}'

    all_matches = list(re.finditer(combined_pattern, text_with_markers))

    if not all_matches or all_matches[0].group(0).strip() == "## END_OF_DOCUMENT##":
        print("⚠️ No se encontraron secciones")
        return []

    for i in range(len(all_matches) - 1):
        start_match = all_matches[i]
        end_match = all_matches[i + 1]

        title = start_match.group(0).strip()
        original_title = next((t for t in section_titles if t.strip().lower() == title.lower()), title)

        start_pos = start_match.end()
        end_pos = end_match.start()
        section_text = text_with_markers[start_pos:end_pos].strip()

        # Extraer referencias citadas
        cited_refs = extract_references(section_text)
        section_resolved_refs = {
            ref_id: resolved_references.get(ref_id, f"Reference [{ref_id}] not found.")
            for ref_id in cited_refs
        }

        sections_data.append({
            "title": original_title,
            "text": section_text,
            "refs": cited_refs,
            "resolved_refs": section_resolved_refs
        })

    return sections_data

def create_chunks_for_chroma(sections_data: List[Dict], paper_name: str, paper_title: str) -> List[Dict]:
    """Crea chunks finos por párrafo para Chroma"""
    chunks = []
    for section in sections_data:
        paragraphs = section['text'].split('\n\n')
        for i, paragraph in enumerate(paragraphs):
            if len(paragraph.strip()) > 50:
                chunk_refs = extract_references(paragraph)
                chunk_resolved_refs = {
                    ref_id: section['resolved_refs'].get(ref_id, f"Ref [{ref_id}] not found")
                    for ref_id in chunk_refs
                }

                chunks.append({
                    "text": paragraph.strip(),
                    "metadata": {
                        "paper_name": paper_name,
                        "paper_title": paper_title,
                        "section_title": section['title'],
                        "chunk_id": f"{paper_name}_{section['title']}_{i}",
                        "references_mentioned": chunk_refs,
                        "resolved_references": chunk_resolved_refs,
                        "chunk_type": "paragraph"
                    }
                })
    return chunks

def create_chunks_for_faiss(sections_data: List[Dict], paper_name: str, paper_title: str) -> List[Dict]:
    """Crea chunks por sección completa para FAISS"""
    return [{
        "text": section['text'],
        "metadata": {
            "paper_name": paper_name,
            "paper_title": paper_title,
            "section_title": section['title'],
            "chunk_id": f"{paper_name}_{section['title']}",
            "references_mentioned": section['refs'],
            "resolved_references": section['resolved_refs'],
            "chunk_type": "full_section"
        }
    } for section in sections_data]

def clean_metadata_for_chroma(metadata: Dict) -> Dict:
    """Limpia metadatos para compatibilidad con Chroma"""
    cleaned = {}
    for key, value in metadata.items():
        if isinstance(value, list):
            cleaned[key] = ", ".join(map(str, value)) if value else ""
        elif isinstance(value, dict):
            cleaned[key] = json.dumps(value) if value else "{}"
        else:
            cleaned[key] = str(value)
    return cleaned

# ============================================================================
# PROCESAMIENTO INDIVIDUAL DE PAPERS CON RATE LIMITING
# ============================================================================

def process_single_paper(pdf_path: str, output_dir: str) -> Dict[str, Any]:
    """Procesa un paper individual y genera sus vectores duales con rate limiting"""

    paper_name = Path(pdf_path).stem
    # Crear nombre sanitizado para colecciones
    sanitized_name = sanitize_collection_name(paper_name)

    print(f"\n🔄 Procesando: {paper_name}")
    print(f"📝 Nombre sanitizado: {sanitized_name}")

    # 1. Extraer texto
    full_text, metadata = extract_pdf_text(pdf_path)
    if full_text is None:
        return {"status": "error", "paper_name": paper_name, "error": "No se pudo extraer texto"}

    print(f"📊 Páginas: {metadata['total_pages']}, Caracteres: {metadata['total_chars']}")

    # 2. Extraer título usando Gemini con rate limiting
    try:
        print("📋 Extrayendo título del paper...")
        paper_title = extract_title_from_text(full_text, gemini_model)
        print(f"📋 Título extraído: {paper_title}")
    except Exception as e:
        print(f"⚠️ Error extrayendo título: {e}")
        paper_title = "Título no identificado"

    # 3. Análisis con Gemini con rate limiting
    try:
        print("🧠 Iniciando análisis con Gemini (con rate limiting)...")
        prompts = create_analysis_prompts_optimized(full_text)

        # Status del rate limiter
        status = rate_limiter.get_status()
        print(f"📊 Rate Limiter Status: RPM {status['rpm_remaining']}/{rate_limiter.rpm_limit}, RPD {status['rpd_remaining']}/{rate_limiter.rpd_limit}")

        # Obtener secciones con rate limiting
        print("  📋 Solicitando secciones...")
        sections_response_text = gemini_request_with_rate_limit(prompts['sections'], "Análisis de secciones")
        sections = parse_gemini_sections(sections_response_text)
        print(f"  📋 Secciones procesadas: {len(sections)}")

        # Obtener referencias con rate limiting
        print("  📚 Solicitando referencias...")
        references_response_text = gemini_request_with_rate_limit(prompts['references'], "Análisis de referencias")
        resolved_references = parse_references_to_dict(references_response_text)
        print(f"  📚 Referencias procesadas: {len(resolved_references)}")

        print(f"✅ Análisis completado - Secciones: {len(sections)}, Referencias: {len(resolved_references)}")

    except Exception as e:
        print(f"❌ Error en análisis Gemini: {e}")
        print(f"🔍 Tipo de error: {type(e).__name__}")
        import traceback
        print(f"🔍 Traceback: {traceback.format_exc()}")
        return {"status": "error", "paper_name": paper_name, "error": f"Error Gemini: {e}"}

    # 4. Procesar secciones
    print("📄 Procesando secciones...")
    sections_data = split_text_by_sections(full_text, sections, resolved_references)

    # 5. Crear chunks
    print("✂️ Creando chunks...")
    chroma_chunks = create_chunks_for_chroma(sections_data, paper_name, paper_title)
    faiss_chunks = create_chunks_for_faiss(sections_data, paper_name, paper_title)

    print(f"📦 Chunks Chroma: {len(chroma_chunks)}, FAISS: {len(faiss_chunks)}")

    # 6. Crear vectorstores
    try:
        print("🔄 Creando vectorstores...")
        # Chroma - usar nombre sanitizado
        chroma_texts = [chunk["text"] for chunk in chroma_chunks]
        chroma_metadatas = [clean_metadata_for_chroma(chunk["metadata"]) for chunk in chroma_chunks]

        collection_name = f"paper_{sanitized_name}_paragraphs"
        print(f"🏷️  Nombre colección Chroma: {collection_name}")

        chroma_store = Chroma.from_texts(
            texts=chroma_texts,
            metadatas=chroma_metadatas,
            embedding=embeddings,
            collection_name=collection_name
        )

        # FAISS
        faiss_texts = [chunk["text"] for chunk in faiss_chunks]
        faiss_metadatas = [chunk["metadata"] for chunk in faiss_chunks]

        faiss_store = FAISS.from_texts(
            texts=faiss_texts,
            metadatas=faiss_metadatas,
            embedding=embeddings
        )

        print("✅ Vectorstores creados")

    except Exception as e:
        print(f"❌ Error creando vectorstores: {e}")
        return {"status": "error", "paper_name": paper_name, "error": f"Error vectorstores: {e}"}

    # 7. Guardar todo
    print("💾 Guardando archivos...")
    paper_dir = os.path.join(output_dir, sanitized_name)
    os.makedirs(paper_dir, exist_ok=True)

    try:
        # Guardar FAISS
        faiss_path = os.path.join(paper_dir, "faiss_index")
        faiss_store.save_local(faiss_path)

        # Guardar Chroma persistente
        chroma_path = os.path.join(paper_dir, "chroma_db")
        chroma_persistent = Chroma(
            collection_name=collection_name,
            embedding_function=embeddings,
            persist_directory=chroma_path
        )
        chroma_persistent.add_texts(texts=chroma_texts, metadatas=chroma_metadatas)

        # Guardar datos procesados
        data_to_save = {
            "paper_name": paper_name,
            "paper_title": paper_title,
            "sanitized_name": sanitized_name,
            "collection_name": collection_name,
            "sections_data": sections_data,
            "sections": sections,
            "resolved_references": resolved_references,
            "metadata": metadata,
            "chroma_chunks_count": len(chroma_chunks),
            "faiss_chunks_count": len(faiss_chunks),
            "rate_limiter_status": rate_limiter.get_status()
        }

        data_path = os.path.join(paper_dir, "processed_data.pkl")
        with open(data_path, 'wb') as f:
            pickle.dump(data_to_save, f)

        print(f"💾 Guardado en: {paper_dir}")

        return {
            "status": "success",
            "paper_name": paper_name,
            "paper_title": paper_title,
            "sanitized_name": sanitized_name,
            "collection_name": collection_name,
            "sections_count": len(sections),
            "references_count": len(resolved_references),
            "chroma_chunks": len(chroma_chunks),
            "faiss_chunks": len(faiss_chunks),
            "output_dir": paper_dir,
            "rate_limiter_status": rate_limiter.get_status()
        }

    except Exception as e:
        print(f"❌ Error guardando: {e}")
        return {"status": "error", "paper_name": paper_name, "error": f"Error guardando: {e}"}

# ============================================================================
# PROCESAMIENTO MASIVO CON CHECKPOINTS
# ============================================================================

def process_all_papers(papers_dir: str, output_base_dir: str) -> Dict[str, Any]:
    """Procesa todos los papers en una carpeta con rate limiting"""

    print("\n🚀 INICIANDO PROCESAMIENTO MASIVO DE PAPERS CON RATE LIMITING")
    print("=" * 80)

    # Verificar que las APIs estén configuradas
    if 'embeddings' not in globals() or 'gemini_model' not in globals():
        print("❌ APIs no configuradas. Ejecuta primero la celda de configuración.")
        return {"status": "error", "message": "APIs no configuradas"}

    print("✅ Usando APIs ya configuradas")

    # Buscar PDFs
    pdf_files = list(Path(papers_dir).glob("*.pdf"))
    print(f"📚 Papers encontrados: {len(pdf_files)}")

    if not pdf_files:
        return {"status": "error", "message": "No se encontraron archivos PDF"}

    # Crear directorio de salida
    os.makedirs(output_base_dir, exist_ok=True)

    # Procesar cada paper
    results = []
    successful = 0
    failed = 0

    for i, pdf_path in enumerate(pdf_files, 1):
        print(f"\n📄 [{i}/{len(pdf_files)}] {pdf_path.name}")
        print("-" * 60)

        # Mostrar status del rate limiter antes de procesar
        status = rate_limiter.get_status()
        print(f"📊 Rate Limiter: RPM {status['rpm_remaining']}/{rate_limiter.rpm_limit}, RPD {status['rpd_remaining']}/{rate_limiter.rpd_limit}")

        # Verificar si podemos continuar
        if status['rpd_remaining'] < 3:  # Menos de 3 requests restantes
            print("⚠️ Límite diario casi alcanzado. Parando procesamiento.")
            break

        result = process_single_paper(str(pdf_path), output_base_dir)
        results.append(result)

        if result["status"] == "success":
            successful += 1
            print(f"✅ Éxito: {result['paper_name']}")
            print(f"📋 Título: {result.get('paper_title', 'No disponible')}")
        else:
            failed += 1
            print(f"❌ Fallo: {result['paper_name']} - {result['error']}")

        # Checkpoint cada 5 papers
        if i % 5 == 0:
            print(f"💾 Checkpoint: {i}/{len(pdf_files)} papers procesados")

    # Guardar resumen general
    summary = {
        "total_papers": len(pdf_files),
        "successful": successful,
        "failed": failed,
        "success_rate": f"{(successful/len(pdf_files)*100):.1f}%",
        "results": results,
        "output_directory": output_base_dir,
        "rate_limiter_final_status": rate_limiter.get_status(),
        "papers_with_titles": [
            {
                "paper_name": r["paper_name"],
                "title": r.get("paper_title", "Título no disponible"),
                "status": r["status"]
            }
            for r in results if r["status"] == "success"
        ]
    }

    summary_path = os.path.join(output_base_dir, "processing_summary_rate_limited.json")
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 80)
    print("📊 RESUMEN FINAL")
    print("=" * 80)
    print(f"📚 Total papers: {summary['total_papers']}")
    print(f"✅ Exitosos: {summary['successful']}")
    print(f"❌ Fallidos: {summary['failed']}")
    print(f"📈 Tasa éxito: {summary['success_rate']}")
    print(f"📁 Directorio salida: {output_base_dir}")
    print(f"📋 Resumen guardado en: {summary_path}")

    # Status final del rate limiter
    final_status = rate_limiter.get_status()
    print(f"\n📊 STATUS FINAL RATE LIMITER:")
    print(f"   🟢 Requests hoy: {final_status['requests_today']}/{rate_limiter.rpd_limit}")
    print(f"   🟡 RPD restantes: {final_status['rpd_remaining']}")

    # Mostrar títulos procesados
    if summary["papers_with_titles"]:
        print(f"\n📋 TÍTULOS EXTRAÍDOS:")
        for paper in summary["papers_with_titles"]:
            print(f"  • {paper['paper_name']}: {paper['title']}")

    return summary

In [10]:
# ============================================================================
# EJECUTAR PROCESAMIENTO CON RATE LIMITING
# ============================================================================

# Configuración de rutas (mantén las que ya tienes configuradas)
papers_dir = "/content/drive/MyDrive/MsC Tesis/Final: Scientific references/papersEVALS"  # 📁 Aquí tienes tus 79 papers
output_base_dir = "/content/drive/MyDrive//MsC Tesis/Final: Scientific references/vectores/multi_papers"  # 📁 Nueva carpeta con rate limiting

print("🎯 CONFIGURACIÓN CON RATE LIMITING:")
print(f"📁 Papers dir: {papers_dir}")
print(f"📁 Output dir: {output_base_dir}")
print(f"🚦 Rate limits: {rate_limiter.rpm_limit} RPM, {rate_limiter.tpm_limit:,} TPM, {rate_limiter.rpd_limit} RPD")

# Verificar que el directorio existe
if not Path(papers_dir).exists():
    print(f"❌ ERROR: Directorio {papers_dir} no existe")
    print("💡 Ajusta la variable 'papers_dir' arriba con tu ruta correcta")
else:
    # Contar PDFs
    pdf_count = len(list(Path(papers_dir).glob("*.pdf")))
    print(f"📊 PDFs encontrados: {pdf_count}")

    if pdf_count == 0:
        print("❌ No se encontraron archivos PDF")
        print("💡 Verifica que la ruta del directorio sea correcta")
    else:
        print(f"\n🚀 Listos para procesar {pdf_count} papers con rate limiting")
        print("⏰ Estimación: ~2-4 minutos por paper debido al rate limiting")
        print("📊 Progreso: Se mostrará en tiempo real")
        print("💾 Checkpoint: Automático cada 5 papers")

        # Status inicial del rate limiter
        status = rate_limiter.get_status()
        print(f"📊 Rate Limiter inicial: RPD {status['rpd_remaining']}/{rate_limiter.rpd_limit}")

        print("\n⚡ EJECUTAR PROCESAMIENTO:")
        print("Descomenta las siguientes líneas para iniciar:")
        print("\n# ===== DESCOMENTA PARA EJECUTAR =====")
        print("# print('🚀 INICIANDO PROCESAMIENTO CON RATE LIMITING...')")
        print("# summary = process_all_papers(papers_dir, output_base_dir)")
        print("#")
        print("# if summary and summary.get('status') != 'error':")
        print("#     print('🎉 ¡PROCESAMIENTO COMPLETADO!')")
        print("#     print(f'📊 Exitosos: {summary[\"successful\"]}/{summary[\"total_papers\"]}')")
        print("#     print(f'📁 Resultados en: {summary[\"output_directory\"]}')")
        print("#     final_status = summary.get('rate_limiter_final_status', {})")
        print("#     print(f'📊 Requests usados: {final_status.get(\"requests_today\", 0)}')")
        print("# else:")
        print("#     print('❌ El procesamiento falló')")
        print("#     if summary:")
        print("#         print(f'Error: {summary.get(\"message\", \"Error desconocido\")}')")
        print("# ===== FIN =====")

        print("\n⚡ EJECUTAR PROCESAMIENTO:")
        print("Descomenta las siguientes líneas para iniciar:")
        print("\n" + "="*50)
        print("# DESCOMENTA PARA EJECUTAR")
        print("="*50)

        # Código de ejecución (descomentado y listo para usar)
        print('🚀 INICIANDO PROCESAMIENTO CON RATE LIMITING...')
        summary = process_all_papers(papers_dir, output_base_dir)
        if summary and summary.get('status') != 'error':
          print('🎉 ¡PROCESAMIENTO COMPLETADO!')
          print(f'📊 Exitosos: {summary["successful"]}/{summary["total_papers"]}')
          print(f'📁 Resultados en: {summary["output_directory"]}')
          final_status = summary.get('rate_limiter_final_status', {})
          print(f'📊 Requests usados: {final_status.get("requests_today", 0)}')
        else:
          print('❌ El procesamiento falló')
          if summary:
            print(f'Error: {summary.get("message", "Error desconocido")}')
            print("="*50)

print("\n📋 CARACTERÍSTICAS DEL SISTEMA CON RATE LIMITING:")
print("• Respeta límites gratuitos de Gemini (15 RPM, 1M TPM, 200 RPD)")
print("• Delay mínimo de 5 segundos entre requests")
print("• Truncamiento inteligente de texto para ahorrar tokens")
print("• Monitoreo en tiempo real del uso de la API")
print("• Para automáticamente si se alcanza el límite diario")
print("• Genera vectorstores Chroma + FAISS para cada paper")
print("• Incluye extracción de títulos y referencias")

🎯 CONFIGURACIÓN CON RATE LIMITING:
📁 Papers dir: /content/drive/MyDrive/MsC Tesis/Final: Scientific references/papersEVALS
📁 Output dir: /content/drive/MyDrive//MsC Tesis/Final: Scientific references/vectores/multi_papers
🚦 Rate limits: 12 RPM, 800,000 TPM, 180 RPD
📊 PDFs encontrados: 42

🚀 Listos para procesar 42 papers con rate limiting
⏰ Estimación: ~2-4 minutos por paper debido al rate limiting
📊 Progreso: Se mostrará en tiempo real
💾 Checkpoint: Automático cada 5 papers
📊 Rate Limiter inicial: RPD 179/180

⚡ EJECUTAR PROCESAMIENTO:
Descomenta las siguientes líneas para iniciar:

# ===== DESCOMENTA PARA EJECUTAR =====
# print('🚀 INICIANDO PROCESAMIENTO CON RATE LIMITING...')
# summary = process_all_papers(papers_dir, output_base_dir)
#
# if summary and summary.get('status') != 'error':
#     print('🎉 ¡PROCESAMIENTO COMPLETADO!')
#     print(f'📊 Exitosos: {summary["successful"]}/{summary["total_papers"]}')
#     print(f'📁 Resultados en: {summary["output_directory"]}')
#     final_st